In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# ==========================================
# 1. Benchmark Functions & Derivatives
# ==========================================

def rosenbrock(x: np.ndarray, a: float = 1.0, b: float = 100.0) -> float:
    """Rosenbrock function: f(x, y) = (a - x)^2 + b * (y - x^2)^2"""
    return (a - x[0]) ** 2 + b * (x[1] - x[0] ** 2) ** 2


def rosenbrock_grad(x: np.ndarray, a: float = 1.0, b: float = 100.0) -> np.ndarray:
    """Gradient of the Rosenbrock function."""
    df_dx = -2 * (a - x[0]) - 4 * b * x[0] * (x[1] - x[0] ** 2)
    df_dy = 2 * b * (x[1] - x[0] ** 2)
    return np.array([df_dx, df_dy])


def rastrigin(x: np.ndarray, A: float = 10.0) -> float:
    """Rastrigin function: Highly non-convex with many local minima."""
    return A * len(x) + np.sum(x ** 2 - A * np.cos(2 * np.pi * x))


def rastrigin_grad(x: np.ndarray, A: float = 10.0) -> np.ndarray:
    """Gradient of the Rastrigin function."""
    return 2 * x + 2 * np.pi * A * np.sin(2 * np.pi * x)


# ==========================================
# 2. Optimization Algorithms
# ==========================================

def gradient_descent(
    grad_fn, init_x: np.ndarray, lr: float = 0.001, num_iterations: int = 2000, tol: float = 1e-6
):
    """Standard Gradient Descent."""
    x = init_x.copy().astype(np.float64)
    history = [x.copy()]
    for _ in range(num_iterations):
        grad = grad_fn(x)
        if np.linalg.norm(grad) < tol:
            break
        x -= lr * grad
        history.append(x.copy())
    return x, history


def momentum_descent(
    grad_fn,
    init_x: np.ndarray,
    lr: float = 0.001,
    beta: float = 0.9,
    num_iterations: int = 2000,
    tol: float = 1e-6,
):
    """Gradient Descent with Polyak Momentum."""
    x = init_x.copy().astype(np.float64)
    v = np.zeros_like(x)
    history = [x.copy()]
    for _ in range(num_iterations):
        grad = grad_fn(x)
        if np.linalg.norm(grad) < tol:
            break
        v = beta * v + lr * grad
        x -= v
        history.append(x.copy())
    return x, history


# ==========================================
# 3. Experiment Execution Engine
# ==========================================

def run_experiment(func, grad_fn, init_x, optimizers):
    """Runs optimizers and records execution time, loss trajectory, and steps taken."""
    results = {}
    for name, opt_fn in optimizers.items():
        start_time = time.perf_counter()
        final_x, trajectory = opt_fn(grad_fn, init_x)
        elapsed_time = (time.perf_counter() - start_time) * 1000.0  # in ms

        loss_history = [func(pt) for pt in trajectory]
        grad_norms = [np.linalg.norm(grad_fn(pt)) for pt in trajectory]

        results[name] = {
            "final_x": final_x,
            "final_loss": loss_history[-1],
            "final_grad_norm": grad_norms[-1],
            "trajectory": np.array(trajectory),
            "loss_history": loss_history,
            "grad_norms": grad_norms,
            "elapsed_ms": elapsed_time,
            "iterations": len(trajectory) - 1,
        }
    return results

In [ ]:
def plot_experiment_results(func, results, bounds, title):
    """Generates contour trajectory plots and log-scale loss convergence graphs."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # --- Plot 1: Contour Plot & Trajectories ---
    x_min, x_max, y_min, y_max = bounds
    X = np.linspace(x_min, x_max, 200)
    Y = np.linspace(y_min, y_max, 200)
    X_grid, Y_grid = np.meshgrid(X, Y)
    Z_grid = np.array([func(np.array([x, y])) for x, y in zip(X_grid.ravel(), Y_grid.ravel())]).reshape(X_grid.shape)

    # Use log scaling for contour levels to handle sharp gradients
    contours = ax1.contour(X_grid, Y_grid, Z_grid, levels=np.logspace(-1, 3, 20), cmap="viridis", alpha=0.6)
    ax1.clabel(contours, inline=True, fontsize=8)

    colors = ["#E63946", "#1D3557", "#2A9D8F", "#F4A261"]
    for idx, (name, res) in enumerate(results.items()):
        traj = res["trajectory"]
        color = colors[idx % len(colors)]
        ax1.plot(traj[:, 0], traj[:, 1], "o-", label=f"{name}", color=color, markersize=2, alpha=0.8)
        ax1.plot(traj[0, 0], traj[0, 1], "go", markersize=7)  # Start Point
        ax1.plot(traj[-1, 0], traj[-1, 1], "rx", markersize=9, markeredgewidth=2)  # End Point

    ax1.set_title(f"{title} - Trajectory Comparison")
    ax1.set_xlabel("$x_1$")
    ax1.set_ylabel("$x_2$")
    ax1.legend()
    ax1.grid(True, linestyle="--", alpha=0.5)

    # --- Plot 2: Convergence Curves ---
    for idx, (name, res) in enumerate(results.items()):
        color = colors[idx % len(colors)]
        ax2.plot(res["loss_history"], label=f"{name}", color=color)

    ax2.set_yscale("log")
    ax2.set_title("Loss Convergence (Log Scale)")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Loss $\mathcal{L}(x)$")
    ax2.legend()
    ax2.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()


def display_summary_table(results):
    """Formats and prints a clean comparison table using pandas."""
    table_data = []
    for name, res in results.items():
        table_data.append({
            "Optimizer": name,
            "Final Loss": f"{res['final_loss']:.6e}",
            "Grad Norm": f"{res['final_grad_norm']:.6e}",
            "Iterations": res["iterations"],
            "Time (ms)": f"{res['elapsed_ms']:.2f}",
            "Final Position [x1, x2]": f"[{res['final_x'][0]:.4f}, {res['final_x'][1]:.4f}]"
        })
    df = pd.DataFrame(table_data)
    display(Markdown(df.to_markdown(index=False)))

In [ ]:
# Configure Experiment 1 Parameters
init_point_rosen = np.array([-1.2, 1.0])
max_iters = 2000
lr_rosen = 0.001

optimizers_rosen = {
    "Standard GD": lambda g, x: gradient_descent(g, x, lr=lr_rosen, num_iterations=max_iters),
    "Momentum (β=0.5)": lambda g, x: momentum_descent(g, x, lr=lr_rosen, beta=0.5, num_iterations=max_iters),
    "Momentum (β=0.9)": lambda g, x: momentum_descent(g, x, lr=lr_rosen, beta=0.9, num_iterations=max_iters),
}

# Run & Display
results_rosen = run_experiment(rosenbrock, rosenbrock_grad, init_point_rosen, optimizers_rosen)
plot_experiment_results(rosenbrock, results_rosen, bounds=(-1.5, 1.5, -0.5, 1.5), title="Rosenbrock Function")
display_summary_table(results_rosen)